# Running a simulation

The notebook version of the page *Running a simulation*. It needs the engine built once (see the repository README); everything below is Python.

Find the top folder of the repository, so that `molsimworkbench` and the `data/` files are found wherever this notebook is opened from.

In [1]:
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "molsimworkbench.py").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "website" / "molsim" / "examples"))
print("working in", root.name)

working in MolSimWorkbench


## Run

The same lines as the page. The engine prints a line every 10 steps. In a notebook those lines reach the cell by a separate route from Python's own `print`, so the two can mix; keeping the result in a variable and printing it in the next cell keeps it readable.

In [2]:
from molsimworkbench import System

sim = System.from_yaml("data/lj_bulk_simulation.yaml",
                       trajectory_output="traj.xyz")
energy = sim.run()

Starting to parse simulation config: data/lj_bulk_simulation.yaml
Successfully parsed JSON from YAML
Extracting simulation type...
Extracting system mode...
System mode set to: lj_particle_box
Extracting file references...
Parsing system mode specific config...
Parsing simulation parameters...
Parsing timestep...
Parsing temperature...
Parsing total steps...
Setting minimization flag...
Parsing minimization algorithm...
Parsing minimization maxIterations...
Parsing minimization energy_tolerance...
Parsing minimization force_tolerance...
Parsing phase control parameters...
Simulation config parsing complete
Creating LJ particle box: N=125, rho=0.8, L=5.38609 Å
Placed 125 particles on cubic lattice


Trajectory output set to traj.xyz (XYZ format), writing every 1 steps.
Initializing velocities for temperature: 1 K
Initial temperature before scaling: 1.08014 K
Initial temperature after scaling: 1 K
Initial kinetic energy: 187.5
Step 0, Potential Energy: -611.553, Kinetic Energy: 187.5, Total Energy: -424.053, Temperature: 1
Step 10, Potential Energy: -608.269, Kinetic Energy: 187.5, Total Energy: -420.769, Temperature: 1
Step 20, Potential Energy: -599.191, Kinetic Energy: 187.5, Total Energy: -411.691, Temperature: 1
Step 30, Potential Energy: -585.872, Kinetic Energy: 187.5, Total Energy: -398.372, Temperature: 1
Step 40, Potential Energy: -569.863, Kinetic Energy: 187.5, Total Energy: -382.363, Temperature: 1
Step 50, Potential Energy: -552.814, Kinetic Energy: 187.5, Total Energy: -365.314, Temperature: 1
Step 60, Potential Energy: -536.268, Kinetic Energy: 187.5, Total Energy: -348.768, Temperature: 1
Step 70, Potential Energy: -522.443, Kinetic Energy: 187.5, Total Energy: -33

4.943, Temperature: 1
Step 80, Potential Energy: -511.296, Kinetic Energy: 187.5, Total Energy: -323.796, Temperature: 1
Step 90, Potential Energy: -503.073, Kinetic Energy: 187.5, Total Energy: -315.573, Temperature: 1
Step 100, Potential Energy: -497.674, Kinetic Energy: 187.5, Total Energy: -310.174, Temperature: 1
Step 110, Potential Energy: -494.578, Kinetic Energy: 187.5, Total Energy: -307.078, Temperature: 1
Step 120, Potential Energy: -493.376, Kinetic Energy: 187.5, Total Energy: -305.876, Temperature: 1
Step 130, Potential Energy: -493.703, Kinetic Energy: 187.5, Total Energy: -306.203, Temperature: 1
Step 140, Potential Energy: -496.638, Kinetic Energy: 187.5, Total Energy: -309.138, Temperature: 1
Step 150, Potential Energy: -502.532, Kinetic Energy: 187.5, Total Energy: -315.032, Temperature: 1
Step 160, Potential Energy: -511.83, Kinetic Energy: 187.5, Total Energy: -324.33, Temperature: 1
Step 170, Potential Energy: -523.006, Kinetic Energy: 187.5, Total Energy: -335.50

perature: 1
Step 680, Potential Energy: -655.443, Kinetic Energy: 187.5, Total Energy: -467.943, Temperature: 1
Step 690, Potential Energy: -656.817, Kinetic Energy: 187.5, Total Energy: -469.317, Temperature: 1
Step 700, Potential Energy: -655.708, Kinetic Energy: 187.5, Total Energy: -468.208, Temperature: 1
Step 710, Potential Energy: -652.23, Kinetic Energy: 187.5, Total Energy: -464.73, Temperature: 1
Step 720, Potential Energy: -646.675, Kinetic Energy: 187.5, Total Energy: -459.175, Temperature: 1
Step 730, Potential Energy: -640.309, Kinetic Energy: 187.5, Total Energy: -452.809, Temperature: 1
Step 740, Potential Energy: -635.931, Kinetic Energy: 187.5, Total Energy: -448.431, Temperature: 1
Step 750, Potential Energy: -636.504, Kinetic Energy: 187.5, Total Energy: -449.004, Temperature: 1
Step 760, Potential Energy: -640.987, Kinetic Energy: 187.5, Total Energy: -453.487, Temperature: 1
Step 770, Potential Energy: -644.821, Kinetic Energy: 187.5, Total Energy: -457.321, Tempe

In [3]:
print("final energy:", energy)

final energy: -458.60397439333326


## Read the trajectory

In [4]:
from read_xyz import read_xyz

x = read_xyz("traj.xyz")
print("frames, atoms, coordinates:", x.shape)

frames, atoms, coordinates: (1002, 125, 3)


## Mean-square displacement

Undo the jumps by one box length, then average the squared distance each atom has moved.

In [5]:
import numpy as np

L, dt = (125 / 0.8) ** (1 / 3), 0.001
step = np.diff(x, axis=0)
step -= L * np.round(step / L)
unwrapped = np.concatenate([x[:1], x[:1] + np.cumsum(step, axis=0)])

for t in (0.25, 0.5, 1.0):
    k = round(t / dt)
    msd = ((unwrapped[k] - unwrapped[0]) ** 2).sum(axis=1).mean()
    print(f"t = {t:4}: MSD = {msd:.3f}")

t = 0.25: MSD = 0.076
t =  0.5: MSD = 0.187
t =  1.0: MSD = 0.416
